# Tuning `tau_edge` on Your Real Data

**Goal:** Find the best cosine-similarity threshold (`tau_edge`) for your specific corpus,
using random seed posts to build intuition about what different threshold values actually *mean*
for your data before running the full sweep.

### Workflow
1. Load your real parquet + embeddings
2. **Seed explorer** — pick a random post, retrieve its nearest neighbours, print text + cosine distance grouped by platform. Repeat until you have a feel for where semantically meaningful similarity starts.
3. **tau sweep** — grid search over candidate thresholds, collecting coverage + coherence metrics
4. Plot tradeoff curves, pick your threshold

---

## 0 · Imports

In [ ]:
# !pip install narrativecluster sentence-transformers matplotlib seaborn scikit-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.preprocessing import normalize as sk_normalize

from narrativecluster import NarrativeClusterer, ClusterConfig

sns.set_theme(style='whitegrid', font_scale=1.15)
pd.set_option('display.max_colwidth', 120)

## 1 · Load your data

Point `PARQUET_PATH` at your deduplicated claim-level parquet.
The only requirement is that it has a text column, a site/platform column,
and a column of pre-computed embeddings (list or np.ndarray per row).

In [ ]:
# ── Configure these three lines for your data ────────────────────────────────
PARQUET_PATH = 'annotation_inputs/df_claim_level.parquet'
TEXT_COL     = 'cleaned_text'
SITE_COL     = 'site'
EMBED_COL    = 'embedding_aligned'
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_parquet(PARQUET_PATH, engine='pyarrow')
print(f'Loaded: {len(df):,} rows')
print(f'Columns: {df.columns.tolist()}')
print(f'\nPlatform breakdown:')
print(df[SITE_COL].value_counts().to_string())

In [ ]:
# Stack embeddings into a matrix and L2-normalise (cosine space)
X = np.vstack(df[EMBED_COL].values).astype(np.float32)
X_norm = sk_normalize(X)   # unit vectors → dot product == cosine similarity
print(f'Embedding matrix: {X_norm.shape}  (N × D)')

# Quick check: cosine sim of a row with itself should be 1.0
assert abs(float(X_norm[0] @ X_norm[0]) - 1.0) < 1e-4, 'Normalisation failed'
print('✅  Normalisation OK')

## 2 · Seed explorer — build intuition before you sweep

Pick a random post as a **seed**, retrieve its top-K nearest neighbours by cosine
similarity, and print them grouped by platform.

This is the most important step — running it 5-10 times with different seeds will
tell you, concretely, what a similarity of 0.70 vs 0.80 vs 0.90 *looks like* in
your corpus.  That intuition directly informs which `tau_edge` is meaningful.

In [ ]:
def explore_seed(
    seed_idx: int,
    df: pd.DataFrame,
    X_norm: np.ndarray,
    text_col: str,
    site_col: str,
    top_k: int = 200,
    n_per_platform: int = 3,
    sim_bins: list = None,
):
    """
    For a given seed post, retrieve top-K neighbours by cosine similarity
    and print a structured report:
      - Seed post text + platform
      - Similarity distribution histogram (text-art)
      - Per-platform sample of n_per_platform posts at each sim_bin threshold
    """
    if sim_bins is None:
        sim_bins = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60]

    seed_vec = X_norm[seed_idx]                        # shape (D,)
    sims     = X_norm @ seed_vec                       # shape (N,) — all cosine sims
    
    # Get top-K excluding self
    order    = np.argsort(-sims)
    order    = order[order != seed_idx][:top_k]
    nbr_sims = sims[order]
    nbr_df   = df.iloc[order].copy()
    nbr_df['_sim'] = nbr_sims

    # ── Seed post ────────────────────────────────────────────────────────────
    seed_row = df.iloc[seed_idx]
    print('=' * 80)
    print(f'SEED  [idx={seed_idx}]  platform={seed_row[site_col]}')
    print(f'  "{seed_row[text_col]}"')
    print('=' * 80)

    # ── Similarity distribution (text histogram) ─────────────────────────────
    print('\nSimilarity distribution among top-K neighbours:')
    edges = np.arange(0.50, 1.01, 0.05)
    counts, _ = np.histogram(nbr_sims, bins=edges)
    max_count = max(counts) if counts.max() > 0 else 1
    for i, (lo, hi, ct) in enumerate(zip(edges[:-1], edges[1:], counts)):
        bar = '█' * int(30 * ct / max_count)
        print(f'  [{lo:.2f}-{hi:.2f})  {bar:<30}  {ct:>4}')
    print()

    # ── Per-platform samples at each threshold ────────────────────────────────
    for threshold in sim_bins:
        band_df = nbr_df[nbr_df['_sim'] >= threshold].copy()
        if len(band_df) == 0:
            continue
        print(f'── sim ≥ {threshold:.2f}  ({len(band_df)} posts) ────────────────────────────')
        for platform, grp in band_df.groupby(site_col):
            sample = grp.nlargest(n_per_platform, '_sim')
            print(f'  [{platform}]  ({len(grp)} total)')
            for _, row in sample.iterrows():
                print(f'    {row["_sim"]:.4f}  {row[text_col][:100]}')
        print()

    return nbr_df

In [ ]:
# ── Run the seed explorer ─────────────────────────────────────────────────────
# Change SEED to any integer, or set to None to pick randomly
SEED = None    # e.g. 42, 1337, None
RNG  = np.random.default_rng(0)

seed_idx = int(RNG.integers(0, len(df))) if SEED is None else SEED
nbr_df   = explore_seed(
    seed_idx     = seed_idx,
    df           = df,
    X_norm       = X_norm,
    text_col     = TEXT_COL,
    site_col     = SITE_COL,
    top_k        = 300,
    n_per_platform = 3,
)

In [ ]:
# ── Run again with a different seed — repeat until you have intuition ─────────
# Each call picks a fresh random post
seed_idx2 = int(RNG.integers(0, len(df)))
_ = explore_seed(seed_idx2, df, X_norm, TEXT_COL, SITE_COL, top_k=300, n_per_platform=3)

In [ ]:
# ── Heatmap: per-platform neighbour density at different sim thresholds ───────
# How many posts from each platform fall above each threshold for the current seed?

thresholds = np.arange(0.55, 0.96, 0.05).round(2)
platforms  = df[SITE_COL].unique()

heat = pd.DataFrame(index=platforms, columns=thresholds, dtype=float)
for tau in thresholds:
    band = nbr_df[nbr_df['_sim'] >= tau]
    vc   = band[SITE_COL].value_counts()
    for p in platforms:
        heat.loc[p, tau] = int(vc.get(p, 0))

fig, ax = plt.subplots(figsize=(13, max(3, len(platforms) * 0.5 + 1)))
sns.heatmap(
    heat.astype(float), annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.4, ax=ax, cbar_kws={'label': '# neighbours'}
)
ax.set_xlabel('tau (cosine similarity threshold)')
ax.set_ylabel('Platform')
ax.set_title(f'Neighbour density by platform & threshold  (seed idx={seed_idx})')
plt.tight_layout()
plt.show()

### What to look for

- Posts that feel semantically equivalent to the seed should be clustering at **your target tau**
- Posts that are only loosely related (same topic, different claim) should fall *below* your tau
- If all neighbours above 0.80 are basically duplicate phrasings → your tau should be lower
- If neighbours at 0.65 are clearly about different things → your tau should be higher

Run the seed explorer on **5–10 diverse posts** across different platforms before committing to a tau grid.

## 3 · Global similarity distribution

Sample random pairs to understand the overall cosine similarity distribution
of your corpus — this anchors the tau grid before the full sweep.

In [ ]:
# Sample 50K random pairs and compute cosine similarities
N_PAIRS = 50_000
rng = np.random.default_rng(42)
idx_a = rng.integers(0, len(df), size=N_PAIRS)
idx_b = rng.integers(0, len(df), size=N_PAIRS)

# mask out self-pairs
ok = idx_a != idx_b
idx_a, idx_b = idx_a[ok], idx_b[ok]

pair_sims = (X_norm[idx_a] * X_norm[idx_b]).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(pair_sims, bins=80, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Cosine similarity'); axes[0].set_ylabel('Count')
axes[0].set_title('Global pairwise similarity (random pairs)')

# CDF — useful for reading off "what fraction of pairs exceed tau"
sorted_sims = np.sort(pair_sims)
cdf = np.arange(1, len(sorted_sims)+1) / len(sorted_sims)
axes[1].plot(sorted_sims, 1 - cdf, color='steelblue', linewidth=2)
axes[1].set_xlabel('tau (cosine similarity threshold)')
axes[1].set_ylabel('Fraction of pairs above tau')
axes[1].set_title('Complementary CDF — fraction of pairs above tau')
axes[1].set_yscale('log')

# Mark some candidate taus
for tau_mark in [0.60, 0.65, 0.70, 0.75, 0.80]:
    frac = float((pair_sims >= tau_mark).mean())
    axes[1].axvline(tau_mark, color='red', alpha=0.4, linestyle='--')
    axes[1].text(tau_mark + 0.003, frac * 1.5, f'{frac:.3f}', fontsize=8, color='red')

plt.tight_layout()
plt.show()

print('Fraction of random pairs above candidate thresholds:')
for tau in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    print(f'  tau={tau:.2f}  →  {(pair_sims >= tau).mean():.4f} of pairs')

## 4 · Fit the clusterer (once)

We fit on a sample (or the full dataset) once, then reuse the HNSW index
for every tau in the sweep — no re-fitting needed.

In [ ]:
# Subsample for speed if your dataset is large
# For the sweep you only need ~10K–50K posts to get stable metrics
SAMPLE_N = min(20_000, len(df))   # set to len(df) to use everything
df_sample = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
print(f'Using {len(df_sample):,} posts for the tau sweep')
print(df_sample[SITE_COL].value_counts().to_string())

In [ ]:
# Set tau_graph based on the CDF above — a good starting point is wherever
# ~1-5% of random pairs are connected (sparse graph → clean Leiden communities)
TAU_GRAPH = 0.68   # adjust based on the CDF plot above

cfg = ClusterConfig(
    embed_col        = EMBED_COL,
    text_col         = TEXT_COL,
    site_col         = SITE_COL,
    k_graph          = 32,
    tau_graph        = TAU_GRAPH,
    cap_m            = 16,
    leiden_resolution= 1.0,
    kq               = 40,
    tau_edge         = 0.65,   # placeholder — swept below
    min_cluster_size = 4,
    query_chunk      = 5_000,
)

CKPT = Path('/tmp/nc_tau_sweep_real')
model = NarrativeClusterer(config=cfg, checkpoint_dir=CKPT)
model.fit(df_sample)

In [ ]:
diag = model.cluster_diagnostics()
print(f'Leiden clusters   : {len(diag):,}')
print(f'Eligible (≥4 posts): {diag["eligible"].sum():,}')
print(f'Median cluster size: {diag["size"].median():.0f}')
print(f'Largest cluster   : {diag["size"].max():,} posts')

## 5 · Sweep `tau_edge` and collect metrics

In [ ]:
def intra_cluster_sim(df_pred, embed_col, max_per_cluster=50, seed=42):
    rng = np.random.default_rng(seed)
    accepted = df_pred[df_pred['nv_accepted']]
    if len(accepted) == 0:
        return np.nan
    sims = []
    for _, grp in accepted.groupby('nv_pred_cluster'):
        if len(grp) < 2:
            continue
        idx  = rng.choice(len(grp), size=min(len(grp), max_per_cluster), replace=False)
        vecs = np.vstack(grp.iloc[idx][embed_col].values).astype(np.float32)
        vecs = sk_normalize(vecs)
        gram = vecs @ vecs.T
        sims.append(float(gram[np.triu_indices(len(vecs), k=1)].mean()))
    return float(np.mean(sims)) if sims else np.nan


# Build tau grid centred on the region that looked interesting in the CDF
TAU_GRID = np.arange(0.55, 0.90, 0.025).round(3).tolist()

records = []
for tau in tqdm(TAU_GRID, desc='tau_edge sweep'):
    model.config.tau_edge = tau
    df_pred  = model.predict(df_sample)
    accepted = df_pred['nv_accepted']

    records.append(dict(
        tau_edge        = tau,
        coverage        = float(accepted.mean()),
        n_assigned      = int(accepted.sum()),
        n_pred_clusters = int(df_pred.loc[accepted, 'nv_pred_cluster'].nunique()) if accepted.any() else 0,
        mean_vote_frac  = float(df_pred.loc[accepted, 'nv_vote_frac'].mean())  if accepted.any() else np.nan,
        mean_best_sim   = float(df_pred.loc[accepted, 'nv_best_sim'].mean())   if accepted.any() else np.nan,
        intra_sim       = intra_cluster_sim(df_pred, EMBED_COL),
    ))

sweep = pd.DataFrame(records)
sweep

## 6 · Visualise tradeoff curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
fig.suptitle('tau_edge sweep on real data', fontsize=14, fontweight='bold')

METRICS = [
    ('coverage',        'Coverage (fraction assigned)',        '#2196F3'),
    ('intra_sim',       'Intra-cluster cosine sim (coherence)','#4CAF50'),
    ('mean_vote_frac',  'Mean vote fraction',                  '#FF9800'),
    ('mean_best_sim',   'Mean best-neighbour sim',             '#9C27B0'),
    ('n_pred_clusters', '# unique predicted clusters',         '#607D8B'),
    ('n_assigned',      '# posts assigned',                    '#E91E63'),
]

elbow_tau = None
for ax, (col, label, color) in zip(axes.flat, METRICS):
    ax.plot(sweep['tau_edge'], sweep[col], marker='o', color=color, linewidth=2)
    ax.set_xlabel('tau_edge')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    if col == 'coverage':
        drops = np.diff(sweep['coverage'].values)
        elbow_idx = int(np.argmin(drops)) + 1
        elbow_tau = float(sweep['tau_edge'].iloc[elbow_idx])
        ax.axvline(elbow_tau, color='red', linestyle='--', alpha=0.7,
                   label=f'Elbow @ {elbow_tau:.3f}')
        ax.legend(fontsize=9)

plt.savefig('/tmp/tau_sweep_real.png', dpi=150)
plt.show()
print(f'Coverage elbow at tau_edge = {elbow_tau:.3f}')

In [ ]:
# Composite score: √(coverage × coherence)
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-12)

sweep['cov_norm']  = minmax(sweep['coverage'])
sweep['coh_norm']  = minmax(sweep['intra_sim'])
sweep['composite'] = np.sqrt(sweep['cov_norm'] * sweep['coh_norm'])

best_row = sweep.loc[sweep['composite'].idxmax()]
best_tau = float(best_row['tau_edge'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep['tau_edge'], sweep['composite'], 'ko-', linewidth=2, label='composite')
ax.plot(sweep['tau_edge'], sweep['cov_norm'],  '--', color='#2196F3', alpha=0.6, label='coverage (norm)')
ax.plot(sweep['tau_edge'], sweep['coh_norm'],  '--', color='#4CAF50', alpha=0.6, label='coherence (norm)')
ax.axvline(best_tau, color='red', linestyle=':', linewidth=1.8, label=f'Best composite: {best_tau:.3f}')
ax.axvline(elbow_tau, color='orange', linestyle=':', linewidth=1.8, label=f'Coverage elbow: {elbow_tau:.3f}')
ax.set_xlabel('tau_edge');  ax.set_ylabel('Score (normalised)')
ax.set_title('Composite = √(coverage × coherence)')
ax.legend();  plt.tight_layout()
plt.savefig('/tmp/tau_composite_real.png', dpi=150)
plt.show()

print(f'✅  Composite-optimal tau_edge = {best_tau:.3f}')
print(f'   Coverage-elbow   tau_edge = {elbow_tau:.3f}')

## 7 · Per-platform acceptance at chosen tau

Different platforms have different posting styles and vocabulary density.
Check whether the global optimum is fair across platforms.

In [ ]:
model.config.tau_edge = best_tau
df_final = model.predict(df_sample)

global_rate, site_tbl = model.site_stats(df_final)
print(f'Global acceptance rate @ tau_edge={best_tau}: {global_rate:.1%}\n')
print(site_tbl[['count', 'accepted_count', 'accept_rate', 'cohens_h', 'effect_size']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#E91E63' if abs(h) >= 0.2 else '#90A4AE' for h in site_tbl['cohens_h'].values]
ax.barh(site_tbl.index, site_tbl['accept_rate'], color=colors)
ax.axvline(global_rate, color='black', linestyle='--', label=f'Global: {global_rate:.1%}')
ax.set_xlabel('Acceptance rate')
ax.set_title('Per-platform acceptance rate (pink = |Cohen h| ≥ 0.2 vs global)')
ax.legend()
plt.tight_layout()
plt.show()

## 8 · Sanity check — spot-inspect accepted assignments

For each platform, print the N most-confidently assigned posts alongside their
predicted cluster, vote fraction, and best-neighbour cosine similarity.

In [ ]:
N_PER_PLATFORM = 4   # number of example posts to show per platform

df_acc = df_final[df_final['nv_accepted']].copy()

for platform, grp in df_acc.groupby(SITE_COL):
    sample = grp.nlargest(N_PER_PLATFORM, 'nv_vote_frac')
    print(f'\n{'─'*70}')
    print(f'Platform: {platform}  ({len(grp):,} accepted / {(df_final[SITE_COL]==platform).sum():,} total)')
    print(f'{'─'*70}')
    for _, row in sample.iterrows():
        print(f'  cluster={row["nv_pred_cluster"]:>5}  '
              f'vote_frac={row["nv_vote_frac"]:.3f}  '
              f'best_sim={row["nv_best_sim"]:.3f}')
        print(f'  "{row[TEXT_COL][:110]}"')
        print()

## 9 · Final recommendation

In [ ]:
print('═' * 60)
print('  tau_edge tuning summary')
print('═' * 60)
print(f'  Composite-optimal : {best_tau:.3f}')
print(f'  Coverage-elbow    : {elbow_tau:.3f}')
print(f'  Global accept rate: {global_rate:.1%}  (@ composite-optimal)')
print()
print('  Suggested ClusterConfig:')
print(f'''
  cfg = ClusterConfig(
      embed_col        = '{EMBED_COL}',
      text_col         = '{TEXT_COL}',
      site_col         = '{SITE_COL}',
      tau_graph        = {TAU_GRAPH},
      tau_edge         = {best_tau:.3f},
      kq               = 50,
      min_cluster_size = 4,
  )
''')
print('  💡 If global acceptance < 30%: lower tau_graph first, not tau_edge')
print('  💡 If per-platform Cohen h is large: consider platform-specific taus')